**本质上是同一种机制：`QMetaObject.invokeMethod` + `Qt.QueuedConnection`。**

`@ensure_main_thread` 的简化实现原理：

```python
def ensure_main_thread(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        # 如果已经在主线程，直接执行
        if QThread.currentThread() == QApplication.instance().thread():
            return func(*args, **kwargs)
        
        # 不在主线程 → 用 QMetaObject.invokeMethod 投递到主线程事件循环
        QMetaObject.invokeMethod(
            wrapper,                          # 目标对象
            "_execute",                       # 要调用的方法
            Qt.QueuedConnection,              # 排队连接（跨线程）
            Q_ARG(...),                       # 带上参数
        )
    return wrapper
```

### 与信号槽对比

```
yield/signal 方式:                      ensure_main_thread 方式:
─────────────────                       ─────────────────────────
GeneratorWorker 内部:                   装饰器内部:
                                        
  self.yielded.emit(10)                 QMetaObject.invokeMethod(
                                          obj, "_execute",
      │                                     Qt.QueuedConnection)
      ▼                                      │
  Qt 内部排入主线程事件队列                   ▼
      │                               Qt 内部排入主线程事件队列
      ▼                                      │
  主线程事件循环取出执行                       ▼
      │                               主线程事件循环取出执行
      ▼                                      │
  connected_slot(10)                         ▼
                                      obj._execute(args)
```

| | 触发方式 | 底层机制 |
|---|---|---|
| `GeneratorWorker.yielded` 信号 | `signal.emit(value)` | Qt 信号槽 → `Qt.QueuedConnection` |
| `@ensure_main_thread` | 函数调用 | `QMetaObject.invokeMethod` → `Qt.QueuedConnection` |

两者最终都是调用同一个 Qt 内部接口：**把一次调用封装为事件，投递到主线程的事件队列**。信号槽多了一层"信号发射 → 槽接收"的抽象，`ensure_main_thread` 更直接：把函数调用本身变成一个排队事件。

所以本质上，`ensure_main_thread` 就是"不需要定义信号，直接让 Qt 帮你排队执行这个函数"。

## 不一定，`@thread_worker` 自动识别两种模式

`@thread_worker` 本质上是**看函数里有没有 `yield` 自动分流**：

| | FunctionWorker | GeneratorWorker |
|---|---|---|
| 函数特征 | 普通函数，**无 `yield`** | 生成器函数，**有 `yield`** |
| 进度回报 | ❌ 不支持 | ✅ `yield` 自动触发 `yielded` 信号 |
| 跨线程 UI 更新 | 需要 `@ensure_main_thread` | 天然安全，无需装饰器 |
| 暂停/恢复/中止 | ❌ 不支持 | ✅ 内置 |
| 可用信号 | `started`, `returned`, `errored` | `yielded`, `returned`, `started`, `errored`, `paused`, `resumed`, `aborted` |

### 实际使用中的比例

看了整个代码库，两种模式都有应用：

**不用 `yield`（FunctionWorker）—— [demo/10_threading.py](file:///Users/usst_ziyi/Programs/trae/Brain/superqt-1/demo/10_threading.py#L99-L108)：**

```python
@thread_worker
def background_task(self):
    for i in range(1, 11):
        time.sleep(0.3)
        self.log(f"后台任务进度: {i * 10}%")          # 需要 @ensure_main_thread
        self.update_progress(i * 10)                  # 需要 @ensure_main_thread
    return "后台任务完成!"
```

适用场景：**纯后台计算，不需要中间回报进度的"一把梭"任务。** 比如"计算一份报表，完成后把结果贴到界面上"。

**用 `yield`（GeneratorWorker）—— [demo/15](file:///Users/usst_ziyi/Programs/trae/Brain/superqt-1/demo/15_superqt_threading_yielded.py)：**

```python
@thread_worker
def task_basic_yield(self, steps: int = 10):
    for i in range(1, steps + 1):
        time.sleep(0.3)
        yield i * (100 // steps)    # 不需要 @ensure_main_thread
    return f"完成 ({steps} 步)"
```

适用场景：**需要实时显示进度、支持用户暂停/取消、或者任务很长需要间歇性回报状态。**

### 一句话总结

> 如果需要"进度条一点点涨"的视觉效果，或者用户要能随时暂停——用 `yield`。  
> 否则 FunctionWorker 更简洁，一个 `return` 收工。